# Experimentação

Este notebook orquestra a Fase 1 e 2 da etapa de experimentação, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [12]:
import sys
from pathlib import Path

# Garante que a raiz do projeto está no sys.path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulação de Dados
import pandas as pd
import numpy as np

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)


from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# Métricas de Avaliação
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer
)


# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.utils.exp import MLPClassifierWrapper

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [13]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [14]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

In [15]:
# Transformação da coluna 'Total Charges' para numérica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [16]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Latitude',
    'Longitude',
    'City',
    'Churn Score',
    'Zip Code',
    'Count'
]

df.drop(columns=drop_cols, inplace=True)

In [17]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e Validação

In [18]:
# Protocolo de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuição da variável alvo nos splits
print("\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===")
print("Treino/Validação:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/Validação: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===
Treino/Validação:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 19)
y_train_val: (4930,)
X_test: (2113, 19)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e Regressão Logística e compara com a MLP e outros modelos de árvores.

In [19]:
# Definindo a etapa de pré-processamento para variáveis categóricas com OHE e numéricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# Dicionário de Pipelines para cada modelo
models = {
    "Dummy": Pipeline([
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", preprocessor),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "DecisionTree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42, class_weight="balanced")),
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight="balanced_subsample"
        )),
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [20]:
# definindo o scoring para avaliação dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [21]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: Dummy ===
=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: DecisionTree ===
=== AVALIANDO MODELO: RandomForest ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6782,0.8576,0.8134,0.5314,0.6427,0.0190,0.0099
1,MLP,0.6728,0.8551,0.8081,0.5139,0.6276,1.0096,0.0113
2,XGBoost,0.6492,0.8438,0.6743,0.5685,0.6167,0.0460,0.0139
3,RandomForest,0.6323,0.8393,0.5091,0.6467,0.5692,0.1559,0.0491
4,DecisionTree,0.3927,0.6684,0.5099,0.5147,0.5120,0.0208,0.0097
5,Dummy,0.2653,0.5000,0.0000,0.0000,0.0000,0.0085,0.0095


### Validando o Wrapper

- Aplicação da MLP fora do pipeline com o ciclo manual de validação para validar o resultado do wrapper.

In [22]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, evaluate, train_with_early_stopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_THRESHOLD = 0.5

ES_PATIENCE = 8
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


mlp_manual_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_raw, y_tr)
    X_va_enc = prep_fold.transform(X_va_raw)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)

    epochs_trained = train_with_early_stopping(
        model,
        train_loader,
        es_loader,
        optimizer,
        criterion,
        device=DEVICE,
        max_epochs=MLP_EPOCHS,
        patience=ES_PATIENCE,
        threshold=MLP_THRESHOLD,
    )
    best_es_loss, _ = evaluate(
        model,
        es_loader,
        criterion,
        device=DEVICE,
        threshold=MLP_THRESHOLD,
    )
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_manual_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": best_es_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

# resultados por fold
mlp_fold_results = pd.DataFrame(mlp_manual_folds)

# médias para comparar com a versão fora do pipeline
mlp_cv_summary = pd.DataFrame([
    {
        "model": "MLP_manual",
        "pr_auc_mean": mlp_fold_results["pr_auc"].mean(),
        "roc_auc_mean": mlp_fold_results["roc_auc"].mean(),
        "recall_mean": mlp_fold_results["recall"].mean(),
        "precision_mean": mlp_fold_results["precision"].mean(),
        "f1_mean": mlp_fold_results["f1"].mean(),
        "fit_time_mean_s": mlp_fold_results["fit_time_s"].mean(),
        "score_time_mean_s": mlp_fold_results["score_time_s"].mean(),
    }
])

#display(mlp_fold_results.round(4))
display(mlp_cv_summary.round(4))



,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_manual,0.6607,0.8528,0.8012,0.5348,0.6406,0.6826,0.0038


### Conclusão

Na tabela de baselines, a `LogisticRegression` apresentou o melhor desempenho geral, com `PR-AUC = 0.6782` e `ROC-AUC = 0.8576`, ficando levemente acima da `MLP` (`PR-AUC = 0.6728` e `ROC-AUC = 0.8551`). Ainda assim, a diferença entre os dois modelos foi pequena, e a `MLP` superou o benchmark de árvore selecionado nesta rodada, o `XGBoost` (`PR-AUC = 0.6492`). Isso indica que, já na base original, a MLP se mostrou competitiva em relação ao baseline linear e ao benchmark não linear.

Na validação da implementação, a `MLP` dentro do `Pipeline` manteve desempenho consistente e até ligeiramente superior à versão manual fora do pipeline, que obteve `PR-AUC = 0.6607` e `ROC-AUC = 0.8528`. Com isso, o wrapper foi validado com sucesso para uso no fluxo de experimentação, trazendo a vantagem de encapsular pré-processamento e validação cruzada dentro da mesma estrutura, com menor risco de leakage e maior facilidade para evoluir o pipeline com feature engineering e seleção de features.


### Logging no MLflow

## Feature Engineering

### Obejtivo

Adicionar poder preditivo aos modelos de forma controlada

In [23]:
# Adicionando as Features


